In [3]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import h5py
import zstandard as zstd
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# ---------- HDF5 metadata ----------
def read_meta_h5(meta_path: Path):
    meta_path = Path(meta_path)
    with h5py.File(meta_path, "r") as f:
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
        data = f["events"][:]   # structured array
    df = pd.DataFrame({
        "x": data["x"], "y": data["y"], "z": data["z"],
        "energy": data["energy"],
        "type_recoil": [s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
                        for s in data["type_recoil"]],
        "no_noise": data["no_noise"],
        "quantize": data["quantize"],
    })
    return attrs, df

# ---------- vectorized unshuffle for a whole batch ----------
def _unshuffle_batch(block: bytes, batch_events: int, n_channels: int, trace_samples: int,
                     dtype=np.float16) -> np.ndarray:
    dtype = np.dtype(dtype)
    itemsize = dtype.itemsize
    num_elements = n_channels * trace_samples

    u8 = np.frombuffer(block, dtype=np.uint8)
    expected = batch_events * itemsize * num_elements
    if u8.size != expected:
        raise ValueError(f"Unexpected batch size: got {u8.size} bytes, expected {expected}")
    u8 = u8.reshape(batch_events, itemsize, num_elements).swapaxes(1, 2).reshape(batch_events, num_elements * itemsize)
    arr = u8.view(dtype)  # (B, N)
    return arr.reshape(batch_events, n_channels, trace_samples)

# ---------- single-file batched iterator (your original) ----------
def iter_traces_zst_batched_merged(traces_path: Path,
                                   n_events: int,
                                   n_channels: int,
                                   trace_samples: int,
                                   batch_size: int = 1000,
                                   dtype=np.float16,
                                   max_events: int | None = None):
    dtype = np.dtype(dtype)
    per_event_bytes = int(n_channels * trace_samples * dtype.itemsize)
    to_read = n_events if max_events is None else min(max_events, n_events)

    dctx = zstd.ZstdDecompressor()
    with open(traces_path, "rb") as fin, dctx.stream_reader(fin) as reader:
        buf = io.BufferedReader(reader)
        remaining = to_read
        while remaining > 0:
            bsz = int(min(batch_size, remaining))
            need = per_event_bytes * bsz
            got, chunks = 0, []
            while got < need:
                chunk = buf.read(need - got)
                if not chunk:
                    raise EOFError(f"Unexpected end of stream: need {need} bytes, got {got} bytes")
                chunks.append(chunk)
                got += len(chunk)
            block = b"".join(chunks)
            yield _unshuffle_batch(block, bsz, n_channels, trace_samples, dtype=dtype)
            remaining -= bsz

# ---------- convenience wrapper (unchanged) ----------
def open_merged_dataset(base_dir: Path, energy: int):
    base = Path(base_dir)
    meta_path = base / f"meta_energy_{energy}.h5"
    traces_path = base / f"traces_energy_{energy}.zst"
    attrs, meta_df = read_meta_h5(meta_path)
    n_events = len(meta_df)
    n_channels = int(attrs["n_channels"])
    trace_samples = int(attrs["trace_samples"])
    trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def get_iter(batch_size=1000, dtype=trace_dtype, max_events=None):
        return iter_traces_zst_batched_merged(traces_path, n_events, n_channels, trace_samples,
                                              batch_size=batch_size, dtype=dtype, max_events=max_events)
    return attrs, meta_df, get_iter

# ---------- PyTorch IterableDataset for the merged stream ----------
class MergedZstIterableDataset(IterableDataset):
    """
    Streams batches from a single merged .zst file.
    Use num_workers = 0 or 1 (compressed stream is inherently sequential).
    Yields CPU torch.float32 tensors shaped (B, C, T).
    """
    def __init__(self, base_dir: Path, energy: int,
                 batch_size: int = 1000,
                 out_dtype=np.float32,
                 max_events: int | None = None):
        super().__init__()
        self.base_dir = Path(base_dir)
        self.energy = energy
        self.batch_size = batch_size
        self.max_events = max_events
        self.out_dtype = np.dtype(out_dtype)

        attrs, meta_df, get_iter = open_merged_dataset(self.base_dir, self.energy)
        self._attrs = attrs
        self._meta_len = len(meta_df)
        self._get_iter = get_iter
        self.n_channels = int(attrs["n_channels"])
        self.trace_samples = int(attrs["trace_samples"])
        self.trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def __iter__(self):
        # Ensure only one worker actually reads; others yield nothing.
        wi = get_worker_info()
        if wi is not None and wi.num_workers > 1 and wi.id != 0:
            return iter(())
        for np_batch in self._get_iter(batch_size=self.batch_size,
                                       dtype=self.trace_dtype,
                                       max_events=self.max_events):
            if self.out_dtype != np_batch.dtype:
                np_batch = np_batch.astype(self.out_dtype, copy=False)
            yield torch.from_numpy(np_batch)

# ---------- Example usage ----------
if __name__ == "__main__":
    BASE = Path("/ceph/dwong/work/training_samples/ER/small")
    E = 1000

    ds = MergedZstIterableDataset(BASE, E, batch_size=200, out_dtype=np.float32, max_events=1000)

    loader = DataLoader(
        ds,
        batch_size=None,          # dataset already yields batches
        num_workers=10,            # keep 0 or 1 for single merged stream
        pin_memory=True,
        prefetch_factor=2,        # ignored when num_workers=0 but fine to leave
        persistent_workers=False
    )

    total = 0
    for cpu_batch in loader:
        # async H2D copy
        batch = cpu_batch.to("cuda", non_blocking=True)
        # ... forward/backward ...
        total += batch.shape[0]
        print(f"batch: {tuple(batch.shape)}, ~{batch.element_size()*batch.nelement()/1024**2:.1f} MB, total: {total}")

    print("Done; read ~", total, "events")


batch: (200, 56, 65536), ~2800.0 MB, total: 200
batch: (200, 56, 65536), ~2800.0 MB, total: 400
batch: (200, 56, 65536), ~2800.0 MB, total: 600
batch: (200, 56, 65536), ~2800.0 MB, total: 800
batch: (200, 56, 65536), ~2800.0 MB, total: 1000
Done; read ~ 1000 events


In [1]:

# pip install nvidia-ml-py3
from pynvml import *
nvmlInit()
gpus = []
for i in range(nvmlDeviceGetCount()):
    h = nvmlDeviceGetHandleByIndex(i)
    name = nvmlDeviceGetName(h).decode()
    mem = nvmlDeviceGetMemoryInfo(h)  # bytes
    gpus.append((i, name, mem.total, mem.used, mem.free))
    print(f"GPU {i} ({name}): total={mem.total/1e9:.2f} GB  used={mem.used/1e9:.2f} GB  free={mem.free/1e9:.2f} GB")
nvmlShutdown()


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.
GPU 0 (NVIDIA L40S): total=48.31 GB  used=0.63 GB  free=47.67 GB
GPU 1 (NVIDIA L40S): total=48.31 GB  used=44.48 GB  free=3.83 GB


In [43]:
#collect all garbage used in current session
torch.cuda.empty_cache()
gc.collect()


9

In [14]:
# ---- Pick a specific GPU by index
gpu_index = 1   # choose whichever GPU you want (0, 1, 2, ...)
device = torch.device(f"cuda:{gpu_index}" if torch.cuda.is_available() else "cpu")

model = ModalityTCXFormer(cfg).to(device)


In [15]:
print("Using device:", device)
print("Device name:", torch.cuda.get_device_name(device))


Using device: cuda:1
Device name: NVIDIA GeForce GTX TITAN X


In [19]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import h5py
import zstandard as zstd
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# -----------------------------
# (1) Your existing helpers
# -----------------------------
def read_meta_h5(meta_path: Path):
    meta_path = Path(meta_path)
    with h5py.File(meta_path, "r") as f:
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
        data = f["events"][:]   # structured array
    df = pd.DataFrame({
        "x": data["x"], "y": data["y"], "z": data["z"],
        "energy": data["energy"],
        "type_recoil": [s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
                        for s in data["type_recoil"]],
        "no_noise": data["no_noise"],
        "quantize": data["quantize"],
    })
    return attrs, df

def _unshuffle_batch(block: bytes, batch_events: int, n_channels: int, trace_samples: int,
                     dtype=np.float16) -> np.ndarray:
    dtype = np.dtype(dtype)
    itemsize = dtype.itemsize
    num_elements = n_channels * trace_samples
    u8 = np.frombuffer(block, dtype=np.uint8)
    expected = batch_events * itemsize * num_elements
    if u8.size != expected:
        raise ValueError(f"Unexpected batch size: got {u8.size} bytes, expected {expected}")
    u8 = u8.reshape(batch_events, itemsize, num_elements).swapaxes(1, 2).reshape(batch_events, num_elements * itemsize)
    arr = u8.view(dtype)  # (B, N)
    return arr.reshape(batch_events, n_channels, trace_samples)

def iter_traces_zst_batched_merged(traces_path: Path,
                                   n_events: int,
                                   n_channels: int,
                                   trace_samples: int,
                                   batch_size: int = 1000,
                                   dtype=np.float16,
                                   max_events: int | None = None):
    dtype = np.dtype(dtype)
    per_event_bytes = int(n_channels * trace_samples * dtype.itemsize)
    to_read = n_events if max_events is None else min(max_events, n_events)

    dctx = zstd.ZstdDecompressor()
    with open(traces_path, "rb") as fin, dctx.stream_reader(fin) as reader:
        buf = io.BufferedReader(reader)
        remaining = to_read
        while remaining > 0:
            bsz = int(min(batch_size, remaining))
            need = per_event_bytes * bsz
            got, chunks = 0, []
            while got < need:
                chunk = buf.read(need - got)
                if not chunk:
                    raise EOFError(f"Unexpected end of stream: need {need} bytes, got {got} bytes")
                chunks.append(chunk)
                got += len(chunk)
            block = b"".join(chunks)
            yield _unshuffle_batch(block, bsz, n_channels, trace_samples, dtype=dtype)
            remaining -= bsz

def open_merged_dataset(base_dir: Path, energy: int):
    base = Path(base_dir)
    meta_path = base / f"meta_energy_{energy}.h5"
    traces_path = base / f"traces_energy_{energy}.zst"
    attrs, meta_df = read_meta_h5(meta_path)
    n_events = len(meta_df)
    n_channels = int(attrs["n_channels"])
    trace_samples = int(attrs["trace_samples"])
    trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def get_iter(batch_size=1000, dtype=trace_dtype, max_events=None):
        return iter_traces_zst_batched_merged(traces_path, n_events, n_channels, trace_samples,
                                              batch_size=batch_size, dtype=dtype, max_events=max_events)
    return attrs, meta_df, get_iter

# -----------------------------
# (2) IterableDataset for training (single merged stream)
# -----------------------------
class MergedTrainingDataset(IterableDataset):
    """
    Streams batches from a single merged .zst file and aligns with metadata rows.
    Yields dicts with tensors:
      - "x": (B, C, T) float32
      - "mask": (B, C) float32 (all ones by default)
      - "pos": (B, 3) float32  (x, y, z)
      - "energy": (B,) float32
      - "cls": (B,) int64  (class index)
    NOTE: Because it is a single compressed stream, keep num_workers=0 (or 1).
    """
    def __init__(self, base_dir: Path, energy: int, batch_size: int = 128,
                 out_dtype=np.float32, max_events: int | None = None):
        super().__init__()
        self.base_dir = Path(base_dir)
        self.energy = energy
        self.batch_size = int(batch_size)
        self.out_dtype = np.dtype(out_dtype)
        self.max_events = max_events

        attrs, meta_df, get_iter = open_merged_dataset(self.base_dir, self.energy)
        self.attrs = attrs
        self.meta_df = meta_df.reset_index(drop=True)
        self.get_iter = get_iter

        self.n_events = len(self.meta_df)
        self.n_channels = int(attrs["n_channels"])
        self.trace_samples = int(attrs["trace_samples"])
        self.in_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

        # Build class map for 'type_recoil'
        classes = sorted(self.meta_df["type_recoil"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.num_classes = len(classes)

    def __iter__(self):
        # If you *really* want multiple workers, make sure only one reads (others return empty)
        wi = get_worker_info()
        if wi is not None and wi.num_workers > 1 and wi.id != 0:
            return iter(())

        meta_idx = 0
        for np_batch in self.get_iter(batch_size=self.batch_size, dtype=self.in_dtype, max_events=self.max_events):
            B = np_batch.shape[0]

            # Convert traces
            if self.out_dtype != np_batch.dtype:
                np_batch = np_batch.astype(self.out_dtype, copy=False)
            x = torch.from_numpy(np_batch)  # (B, C, T), float32

            # Channel mask (all channels active unless you have per-event masking)
            mask = torch.ones((B, self.n_channels), dtype=torch.float32)

            # Slice matching metadata
            meta_slice = self.meta_df.iloc[meta_idx: meta_idx + B]
            meta_idx += B

            pos = torch.tensor(meta_slice[["x", "y", "z"]].to_numpy(), dtype=torch.float32)        # (B,3)
            energy = torch.tensor(meta_slice["energy"].to_numpy(), dtype=torch.float32)            # (B,)
            cls = torch.tensor([self.class_to_idx[s] for s in meta_slice["type_recoil"]], dtype=torch.long)

            yield {"x": x, "mask": mask, "pos": pos, "energy": energy, "cls": cls}

# -----------------------------
# (3) Training loop wiring
# -----------------------------
def human_mb(t: torch.Tensor) -> float:
    return (t.element_size() * t.nelement()) / (1024**2)

def train_one_epoch(model, loader, optimizer, scaler, device,
                    w_pos=1.0, w_energy=1.0, w_cls=1.0):
    model.train()
    loss_pos_fn = nn.SmoothL1Loss()      # robust for coordinates
    loss_energy_fn = nn.MSELoss()        # or SmoothL1Loss
    loss_cls_fn = nn.CrossEntropyLoss()

    running = {"loss": 0.0, "pos": 0.0, "energy": 0.0, "cls": 0.0}
    steps = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)         # (B,C,T)
        mask = batch["mask"].to(device, non_blocking=True)   # (B,C)
        y_pos = batch["pos"].to(device, non_blocking=True)   # (B,3)
        y_energy = batch["energy"].to(device, non_blocking=True)  # (B,)
        y_cls = batch["cls"].to(device, non_blocking=True)   # (B,)

        with torch.cuda.amp.autocast():
            pos_pred, energy_pred, cls_logit, aux = model(x, channel_mask=mask)

            # Ensure shapes align (squeeze if the model returns (B,1))
            if energy_pred.dim() == 2 and energy_pred.shape[1] == 1:
                energy_pred = energy_pred.squeeze(1)

            loss_pos = loss_pos_fn(pos_pred, y_pos)
            loss_energy = loss_energy_fn(energy_pred, y_energy)
            loss_cls = loss_cls_fn(cls_logit, y_cls)

            loss = w_pos * loss_pos + w_energy * loss_energy + w_cls * loss_cls

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running["loss"] += loss.item()
        running["pos"] += loss_pos.item()
        running["energy"] += loss_energy.item()
        running["cls"] += loss_cls.item()
        steps += 1

        # Quick and dirty progress
        print(f"batch {steps:5d}: "
              f"loss={loss.item():.4f} "
              f"(pos={loss_pos.item():.4f}, E={loss_energy.item():.4f}, cls={loss_cls.item():.4f}) "
              f"x={tuple(x.shape)} ~{human_mb(x):.1f}MB")

    for k in running:
        running[k] /= max(steps, 1)
    return running

# -----------------------------
# (4) Main: build model + run
# -----------------------------
if __name__ == "__main__":
    from tcxformer_v2 import ModelConfig, ModalityTCXFormer  # your file

    torch.manual_seed(0)
    device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

    # ---- Config (use your values)
    cfg = ModelConfig(
        n_photon=19,
        n_phonon=37,
        stride_photon=128,
        stride_phonon=256,
        d_model=256,
        d_ff=1024,
        n_heads=4,
        n_time_layers=3,
        n_chan_layers=1,
        patch_embed=64,
        dropout=0.1,
        rope_base=10000.0,
        n_branch_to_task=2,
    )
    model = ModalityTCXFormer(cfg).to(device)

    # ---- Data
    BASE = Path("/ceph/dwong/work/training_samples/ER/small")
    ENERGY_BIN = 1000

    # Important: single merged stream → keep num_workers=0 (sequential)
    ds = MergedTrainingDataset(BASE, ENERGY_BIN, batch_size=64, out_dtype=np.float32, max_events=None)
    print(f"Classes: {ds.class_to_idx}  (num={ds.num_classes})")
    loader = DataLoader(
        ds,
        batch_size=None,      # dataset yields ready-made batches
        num_workers=1,        # keep 0/1 for a single merged .zst
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2,    # ignored with num_workers=0 but harmless
    )

    # ---- Optimizer / AMP
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    # ---- Train a few epochs
    for epoch in range(1, 4):
        stats = train_one_epoch(model, loader, optimizer, scaler, device,
                                w_pos=1.0, w_energy=1.0, w_cls=1.0)
        print(f"[epoch {epoch}] "
              f"loss={stats['loss']:.4f}  pos={stats['pos']:.4f}  "
              f"E={stats['energy']:.4f}  cls={stats['cls']:.4f}")


Classes: {'ER': 0}  (num=1)


/tmp/ipykernel_3406071/2874186881.py:254: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_3406071/2874186881.py:175: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.75 GiB. GPU 3 has a total capacity of 11.92 GiB of which 1.36 GiB is free. Including non-PyTorch memory, this process has 10.53 GiB memory in use. Of the allocated memory 9.71 GiB is allocated by PyTorch, and 136.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [36]:
from pathlib import Path
import io, gc
import numpy as np
import pandas as pd
import h5py
import zstandard as zstd
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# -----------------------------
# (1) Helpers (unchanged)
# -----------------------------
def read_meta_h5(meta_path: Path):
    meta_path = Path(meta_path)
    with h5py.File(meta_path, "r") as f:
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
        data = f["events"][:]   # structured array
    df = pd.DataFrame({
        "x": data["x"], "y": data["y"], "z": data["z"],
        "energy": data["energy"],
        "type_recoil": [s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
                        for s in data["type_recoil"]],
        "no_noise": data["no_noise"],
        "quantize": data["quantize"],
    })
    return attrs, df

def _unshuffle_batch(block: bytes, batch_events: int, n_channels: int, trace_samples: int,
                     dtype=np.float16) -> np.ndarray:
    dtype = np.dtype(dtype)
    itemsize = dtype.itemsize
    num_elements = n_channels * trace_samples
    u8 = np.frombuffer(block, dtype=np.uint8)
    expected = batch_events * itemsize * num_elements
    if u8.size != expected:
        raise ValueError(f"Unexpected batch size: got {u8.size} bytes, expected {expected}")
    u8 = u8.reshape(batch_events, itemsize, num_elements).swapaxes(1, 2).reshape(batch_events, num_elements * itemsize)
    arr = u8.view(dtype)  # (B, N)
    return arr.reshape(batch_events, n_channels, trace_samples)

def iter_traces_zst_batched_merged(traces_path: Path,
                                   n_events: int,
                                   n_channels: int,
                                   trace_samples: int,
                                   batch_size: int = 200,
                                   dtype=np.float16,
                                   max_events: int | None = None):
    dtype = np.dtype(dtype)
    per_event_bytes = int(n_channels * trace_samples * dtype.itemsize)
    to_read = n_events if max_events is None else min(max_events, n_events)

    dctx = zstd.ZstdDecompressor()
    with open(traces_path, "rb") as fin, dctx.stream_reader(fin) as reader:
        buf = io.BufferedReader(reader)
        remaining = to_read
        while remaining > 0:
            bsz = int(min(batch_size, remaining))
            need = per_event_bytes * bsz
            got, chunks = 0, []
            while got < need:
                chunk = buf.read(need - got)
                if not chunk:
                    raise EOFError(f"Unexpected end of stream: need {need} bytes, got {got} bytes")
                chunks.append(chunk)
                got += len(chunk)
            block = b"".join(chunks)
            yield _unshuffle_batch(block, bsz, n_channels, trace_samples, dtype=dtype)
            remaining -= bsz

def open_merged_dataset(base_dir: Path, energy: int):
    base = Path(base_dir)
    meta_path = base / f"meta_energy_{energy}.h5"
    traces_path = base / f"traces_energy_{energy}.zst"
    attrs, meta_df = read_meta_h5(meta_path)
    n_events = len(meta_df)
    n_channels = int(attrs["n_channels"])
    trace_samples = int(attrs["trace_samples"])
    trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def get_iter(batch_size=200, dtype=trace_dtype, max_events=None):
        return iter_traces_zst_batched_merged(traces_path, n_events, n_channels, trace_samples,
                                              batch_size=batch_size, dtype=dtype, max_events=max_events)
    return attrs, meta_df, get_iter

# -----------------------------
# (2) IterableDataset
# -----------------------------
class MergedTrainingDataset(IterableDataset):
    """
    Streams batches from a single merged .zst file and aligns with metadata rows.
    Yields dicts:
      - "x": (B, C, T) float32
      - "mask": (B, C) float32
      - "pos": (B, 3) float32
      - "energy": (B,) float32
      - "cls": (B,) int64
    """
    def __init__(self, base_dir: Path, energy: int, batch_size: int = 128,
                 out_dtype=np.float32, max_events: int | None = None):
        super().__init__()
        self.base_dir = Path(base_dir)
        self.energy = energy
        self.batch_size = int(batch_size)
        self.out_dtype = np.dtype(out_dtype)
        self.max_events = max_events

        attrs, meta_df, get_iter = open_merged_dataset(self.base_dir, self.energy)
        self.attrs = attrs
        self.meta_df = meta_df.reset_index(drop=True)
        self.get_iter = get_iter

        self.n_events = len(self.meta_df)
        self.n_channels = int(attrs["n_channels"])
        self.trace_samples = int(attrs["trace_samples"])
        self.in_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

        # class map
        classes = sorted(self.meta_df["type_recoil"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.num_classes = len(classes)

    def __iter__(self):
        wi = get_worker_info()
        if wi is not None and wi.num_workers > 1 and wi.id != 0:
            return iter(())  # only worker 0 reads the single stream

        meta_idx = 0
        for np_batch in self.get_iter(batch_size=self.batch_size, dtype=self.in_dtype, max_events=self.max_events):
            B = np_batch.shape[0]

            if self.out_dtype != np_batch.dtype:
                np_batch = np_batch.astype(self.out_dtype, copy=False)
            x = torch.from_numpy(np_batch)  # (B, C, T)

            mask = torch.ones((B, self.n_channels), dtype=torch.float32)

            meta_slice = self.meta_df.iloc[meta_idx: meta_idx + B]
            meta_idx += B

            pos = torch.tensor(meta_slice[["x", "y", "z"]].to_numpy(), dtype=torch.float32)
            energy = torch.tensor(meta_slice["energy"].to_numpy(), dtype=torch.float32)
            cls = torch.tensor([self.class_to_idx[s] for s in meta_slice["type_recoil"]], dtype=torch.long)

            yield {"x": x, "mask": mask, "pos": pos, "energy": energy, "cls": cls}

# -----------------------------
# (3) Training utils
# -----------------------------
def human_mb(t: torch.Tensor) -> float:
    return (t.element_size() * t.nelement()) / (1024**2)

@torch.no_grad()
def _trim_cache_every(n, step):
    # Call occasionally to combat fragmentation; adjust cadence if needed
    if step % n == 0:
        torch.cuda.empty_cache()

def train_one_epoch(model, loader, optimizer, scaler, device,
                    w_pos=1.0, w_energy=1.0, w_cls=1.0, cache_trim_every=50):
    model.train()
    loss_pos_fn = nn.SmoothL1Loss()
    loss_energy_fn = nn.MSELoss()
    loss_cls_fn = nn.CrossEntropyLoss()

    running = {"loss": 0.0, "pos": 0.0, "energy": 0.0, "cls": 0.0}
    steps = 0

    for batch in loader:
        # unpack & drop the dict ASAP (avoid keeping refs)
        x_cpu = batch["x"]; mask_cpu = batch["mask"]
        y_pos_cpu = batch["pos"]; y_energy_cpu = batch["energy"]; y_cls_cpu = batch["cls"]
        del batch

        x = x_cpu.to(device, non_blocking=True)
        mask = mask_cpu.to(device, non_blocking=True)
        y_pos = y_pos_cpu.to(device, non_blocking=True)
        y_energy = y_energy_cpu.to(device, non_blocking=True)
        y_cls = y_cls_cpu.to(device, non_blocking=True)
        del x_cpu, mask_cpu, y_pos_cpu, y_energy_cpu, y_cls_cpu

        with torch.cuda.amp.autocast():
            pos_pred, energy_pred, cls_logit, aux = model(x, channel_mask=mask)
            if energy_pred.dim() == 2 and energy_pred.shape[1] == 1:
                energy_pred = energy_pred.squeeze(1)

            loss_pos = loss_pos_fn(pos_pred, y_pos)
            loss_energy = loss_energy_fn(energy_pred, y_energy)
            loss_cls = loss_cls_fn(cls_logit, y_cls)
            loss = w_pos * loss_pos + w_energy * loss_energy + w_cls * loss_cls

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running["loss"] += loss.item()
        running["pos"] += loss_pos.item()
        running["energy"] += loss_energy.item()
        running["cls"] += loss_cls.item()
        steps += 1

        print(f"batch {steps:5d}: "
              f"loss={loss.item():.4f} "
              f"(pos={loss_pos.item():.4f}, E={loss_energy.item():.4f}, cls={loss_cls.item():.4f}) "
              f"x={tuple(x.shape)} ~{human_mb(x):.1f}MB")

        # ---- explicit per-iter cleanup (drop GPU refs) ----
        del x, mask, y_pos, y_energy, y_cls
        del pos_pred, energy_pred, cls_logit, aux, loss_pos, loss_energy, loss_cls, loss
        _trim_cache_every(cache_trim_every, steps)

    for k in running:
        running[k] /= max(steps, 1)
    return running

# -----------------------------
# (4) Main
# -----------------------------
if __name__ == "__main__":
    from tcxformer_v2 import ModelConfig, ModalityTCXFormer  # your file

    torch.manual_seed(0)

    # ---- Choose GPU explicitly
    gpu_index = 1  # <--- pick one
    device = torch.device(f"cuda:{gpu_index}" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    if device.type == "cuda":
        torch.cuda.set_device(device)  # set default device

    # (Optional) cudnn settings — can reduce transient workspace sizes
    torch.backends.cudnn.benchmark = False

    # ---- Model
    cfg = ModelConfig(
        n_photon=19, n_phonon=37,
        stride_photon=128, stride_phonon=256,
        d_model=256, d_ff=1024, n_heads=4,
        n_time_layers=3, n_chan_layers=1,
        patch_embed=64, dropout=0.1,
        rope_base=10000.0, n_branch_to_task=2,
    )
    model = ModalityTCXFormer(cfg).to(device)

    # ---- Data
    BASE = Path("/ceph/dwong/work/training_samples/ER/small")
    ENERGY_BIN = 1000
    ds = MergedTrainingDataset(BASE, ENERGY_BIN, batch_size=64, out_dtype=np.float32, max_events=None)
    print(f"Classes: {ds.class_to_idx}  (num={ds.num_classes})")

    loader = DataLoader(
        ds,
        batch_size=None,       # dataset yields ready-made batches
        num_workers=1,         # keep 0/1 for single merged .zst
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2,
    )

    # ---- Optimizer / AMP
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    try:
        for epoch in range(1, 4):
            stats = train_one_epoch(model, loader, optimizer, scaler, device,
                                    w_pos=1.0, w_energy=1.0, w_cls=1.0,
                                    cache_trim_every=50)
            print(f"[epoch {epoch}] "
                  f"loss={stats['loss']:.4f}  pos={stats['pos']:.4f}  "
                  f"E={stats['energy']:.4f}  cls={stats['cls']:.4f}")
    finally:
        # ---- aggressive cleanup (useful in notebooks) ----
        del model, optimizer, scaler, loader, ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


Using device: cuda:1
Classes: {'ER': 0}  (num=1)


/tmp/ipykernel_3406071/4229581576.py:263: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_3406071/4229581576.py:182: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.75 GiB. GPU 1 has a total capacity of 11.92 GiB of which 1.38 GiB is free. Including non-PyTorch memory, this process has 10.51 GiB memory in use. Of the allocated memory 9.71 GiB is allocated by PyTorch, and 116.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [1]:
from pathlib import Path
import os, io, gc
import numpy as np
import pandas as pd
import h5py
import zstandard as zstd
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# -----------------------------
# (0) OOM-friendly runtime knobs
# -----------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True           # Ampere+
torch.set_float32_matmul_precision("medium")           # PyTorch 2.x

# -----------------------------
# (1) Helpers (unchanged)
# -----------------------------
def read_meta_h5(meta_path: Path):
    meta_path = Path(meta_path)
    with h5py.File(meta_path, "r") as f:
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
        data = f["events"][:]   # structured array
    df = pd.DataFrame({
        "x": data["x"], "y": data["y"], "z": data["z"],
        "energy": data["energy"],
        "type_recoil": [s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
                        for s in data["type_recoil"]],
        "no_noise": data["no_noise"],
        "quantize": data["quantize"],
    })
    return attrs, df

def _unshuffle_batch(block: bytes, batch_events: int, n_channels: int, trace_samples: int,
                     dtype=np.float16) -> np.ndarray:
    dtype = np.dtype(dtype)
    itemsize = dtype.itemsize
    num_elements = n_channels * trace_samples
    u8 = np.frombuffer(block, dtype=np.uint8)
    expected = batch_events * itemsize * num_elements
    if u8.size != expected:
        raise ValueError(f"Unexpected batch size: got {u8.size} bytes, expected {expected}")
    u8 = u8.reshape(batch_events, itemsize, num_elements).swapaxes(1, 2).reshape(batch_events, num_elements * itemsize)
    arr = u8.view(dtype)  # (B, N)
    return arr.reshape(batch_events, n_channels, trace_samples)

def iter_traces_zst_batched_merged(traces_path: Path,
                                   n_events: int,
                                   n_channels: int,
                                   trace_samples: int,
                                   batch_size: int = 200,
                                   dtype=np.float16,
                                   max_events: int | None = None):
    dtype = np.dtype(dtype)
    per_event_bytes = int(n_channels * trace_samples * dtype.itemsize)
    to_read = n_events if max_events is None else min(max_events, n_events)

    dctx = zstd.ZstdDecompressor()
    with open(traces_path, "rb") as fin, dctx.stream_reader(fin) as reader:
        buf = io.BufferedReader(reader)
        remaining = to_read
        while remaining > 0:
            bsz = int(min(batch_size, remaining))
            need = per_event_bytes * bsz
            got, chunks = 0, []
            while got < need:
                chunk = buf.read(need - got)
                if not chunk:
                    raise EOFError(f"Unexpected end of stream: need {need} bytes, got {got} bytes")
                chunks.append(chunk)
                got += len(chunk)
            block = b"".join(chunks)
            yield _unshuffle_batch(block, bsz, n_channels, trace_samples, dtype=dtype)
            remaining -= bsz

def open_merged_dataset(base_dir: Path, energy: int):
    base = Path(base_dir)
    meta_path = base / f"meta_energy_{energy}.h5"
    traces_path = base / f"traces_energy_{energy}.zst"
    attrs, meta_df = read_meta_h5(meta_path)
    n_events = len(meta_df)
    n_channels = int(attrs["n_channels"])
    trace_samples = int(attrs["trace_samples"])
    trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def get_iter(batch_size=200, dtype=trace_dtype, max_events=None):
        return iter_traces_zst_batched_merged(traces_path, n_events, n_channels, trace_samples,
                                              batch_size=batch_size, dtype=dtype, max_events=max_events)
    return attrs, meta_df, get_iter

# -----------------------------
# (2) IterableDataset
# -----------------------------
class MergedTrainingDataset(IterableDataset):
    """
    Streams batches from a single merged .zst file and aligns with metadata rows.
    Yields dicts:
      - "x": (B, C, T) float32
      - "mask": (B, C) float32
      - "pos": (B, 3) float32
      - "energy": (B,) float32
      - "cls": (B,) int64
    """
    def __init__(self, base_dir: Path, energy: int, batch_size: int = 64,
                 out_dtype=np.float32, max_events: int | None = None):
        super().__init__()
        self.base_dir = Path(base_dir)
        self.energy = energy
        self.batch_size = int(batch_size)
        self.out_dtype = np.dtype(out_dtype)
        self.max_events = max_events

        attrs, meta_df, get_iter = open_merged_dataset(self.base_dir, self.energy)
        self.attrs = attrs
        self.meta_df = meta_df.reset_index(drop=True)
        self.get_iter = get_iter

        self.n_events = len(self.meta_df)
        self.n_channels = int(attrs["n_channels"])
        self.trace_samples = int(attrs["trace_samples"])
        self.in_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

        classes = sorted(self.meta_df["type_recoil"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.num_classes = len(classes)

    def __iter__(self):
        wi = get_worker_info()
        if wi is not None and wi.num_workers > 1 and wi.id != 0:
            return iter(())  # only worker 0 reads the single stream

        meta_idx = 0
        for np_batch in self.get_iter(batch_size=self.batch_size, dtype=self.in_dtype, max_events=self.max_events):
            B = np_batch.shape[0]

            if self.out_dtype != np_batch.dtype:
                np_batch = np_batch.astype(self.out_dtype, copy=False)
            x = torch.from_numpy(np_batch)  # (B, C, T)

            mask = torch.ones((B, self.n_channels), dtype=torch.float32)

            meta_slice = self.meta_df.iloc[meta_idx: meta_idx + B]
            meta_idx += B

            pos = torch.tensor(meta_slice[["x", "y", "z"]].to_numpy(), dtype=torch.float32)
            energy = torch.tensor(meta_slice["energy"].to_numpy(), dtype=torch.float32)
            cls = torch.tensor([self.class_to_idx[s] for s in meta_slice["type_recoil"]], dtype=torch.long)

            yield {"x": x, "mask": mask, "pos": pos, "energy": energy, "cls": cls}

# -----------------------------
# (3) Training utils
# -----------------------------
def human_mb(t: torch.Tensor) -> float:
    return (t.element_size() * t.nelement()) / (1024**2)

@torch.no_grad()
def _trim_cache_every(n, step):
    if step % n == 0:
        torch.cuda.empty_cache()

def train_one_epoch(model, loader, optimizer, scaler, device,
                    w_pos=1.0, w_energy=1.0, w_cls=1.0,
                    accum_steps=4, cache_trim_every=50):
    """Gradient accumulation to reduce peak VRAM.
       Effective batch = dataset_batch_size, but we split it into microbatches on GPU."""
    model.train()
    loss_pos_fn = nn.SmoothL1Loss()
    loss_energy_fn = nn.MSELoss()
    loss_cls_fn = nn.CrossEntropyLoss()

    running = {"loss": 0.0, "pos": 0.0, "energy": 0.0, "cls": 0.0}
    steps = 0

    optimizer.zero_grad(set_to_none=True)

    for batch in loader:
        # CPU tensors
        x_cpu = batch["x"]; m_cpu = batch["mask"]
        yp_cpu = batch["pos"]; ye_cpu = batch["energy"]; yc_cpu = batch["cls"]
        del batch

        B = x_cpu.shape[0]
        # derive a microbatch size; try to split B into ~accum_steps parts
        mb = max(1, B // accum_steps)
        if B % mb != 0:
            mb = 1  # fallback to 1 if it doesn't divide nicely

        # accumulate gradients over microbatches
        for i in range(0, B, mb):
            x = x_cpu[i:i+mb].to(device, non_blocking=True)
            m = m_cpu[i:i+mb].to(device, non_blocking=True)
            yp = yp_cpu[i:i+mb].to(device, non_blocking=True)
            ye = ye_cpu[i:i+mb].to(device, non_blocking=True)
            yc = yc_cpu[i:i+mb].to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                pos_pred, energy_pred, cls_logit, aux = model(x, channel_mask=m)
                if energy_pred.dim() == 2 and energy_pred.shape[1] == 1:
                    energy_pred = energy_pred.squeeze(1)

                loss_pos = loss_pos_fn(pos_pred, yp)
                loss_energy = loss_energy_fn(energy_pred, ye)
                loss_cls = loss_cls_fn(cls_logit, yc)
                loss = w_pos * loss_pos + w_energy * loss_energy + w_cls * loss_cls

                # scale loss so summed grads equal full-batch grads
                loss = loss * (mb / B)

            scaler.scale(loss).backward()

            # free microbatch temporaries
            del x, m, yp, ye, yc, pos_pred, energy_pred, cls_logit, aux, loss_pos, loss_energy, loss_cls, loss

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        # light logging (avoid keeping tensors)
        steps += 1
        if steps % cache_trim_every == 0:
            torch.cuda.empty_cache()

        # free CPU batch tensors
        del x_cpu, m_cpu, yp_cpu, ye_cpu, yc_cpu

    # (optional) compute epoch averages if you log per-microbatch losses above
    for k in running:
        running[k] /= max(steps, 1)
    return running

# -----------------------------
# (4) Main
# -----------------------------
if __name__ == "__main__":
    from tcxformer_v2 import ModelConfig, ModalityTCXFormer  # your file

    torch.manual_seed(0)

    # ---- Choose GPU explicitly
    gpu_index = 2
    device = torch.device(f"cuda:{gpu_index}" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    if device.type == "cuda":
        torch.cuda.set_device(device)

    # ---- Model
    cfg = ModelConfig(
        n_photon=19, n_phonon=37,
        stride_photon=128, stride_phonon=256,
        d_model=256, d_ff=1024, n_heads=4,
        n_time_layers=3, n_chan_layers=1,
        patch_embed=64, dropout=0.1,
        rope_base=10000.0, n_branch_to_task=2,
    )
    model = ModalityTCXFormer(cfg).to(device)

    # ---- Data
    BASE = Path("/ceph/dwong/work/training_samples/ER/small")
    ENERGY_BIN = 1000
    # keep dataset batch modest; accum_steps handles the rest on GPU
    ds = MergedTrainingDataset(BASE, ENERGY_BIN, batch_size=64, out_dtype=np.float32, max_events=None)
    print(f"Classes: {ds.class_to_idx}  (num={ds.num_classes})")

    loader = DataLoader(
        ds,
        batch_size=None,       # dataset yields ready-made batches
        num_workers=1,         # keep 0/1 for single merged .zst
        pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2,
    )

    # ---- Optimizer / AMP
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    try:
        for epoch in range(1, 4):
            stats = train_one_epoch(
                model, loader, optimizer, scaler, device,
                w_pos=1.0, w_energy=1.0, w_cls=1.0,
                accum_steps=16,              # <-- increase if still OOM (e.g., 8, 12, 16)
                cache_trim_every=50
            )
            print(f"[epoch {epoch}] done")
    finally:
        # Aggressive cleanup (useful in notebooks/REPL)
        del model, optimizer, scaler, loader, ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


/home/dwong/anaconda3/envs/icl/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Using device: cuda:2
Classes: {'ER': 0}  (num=1)


/tmp/ipykernel_3462909/2118519162.py:279: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_3462909/2118519162.py:200: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cpu, different from other tensors on cuda:2 (when checking argument in method wrapper_CUDA_mm)

In [ ]:
from pathlib import Path
import os, io, gc
import numpy as np
import pandas as pd
import h5py
import zstandard as zstd
import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# -----------------------------
# (0) OOM-friendly runtime knobs
# -----------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True           # Ampere+
try:
    torch.set_float32_matmul_precision("medium")       # PyTorch 2.x
except Exception:
    pass

# -----------------------------
# (1) Helpers (unchanged)
# -----------------------------
def read_meta_h5(meta_path: Path):
    meta_path = Path(meta_path)
    with h5py.File(meta_path, "r") as f:
        attrs = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in f.attrs.items()}
        data = f["events"][:]   # structured array
    df = pd.DataFrame({
        "x": data["x"], "y": data["y"], "z": data["z"],
        "energy": data["energy"],
        "type_recoil": [s.decode("utf-8") if isinstance(s, (bytes, bytearray)) else str(s)
                        for s in data["type_recoil"]],
        "no_noise": data["no_noise"],
        "quantize": data["quantize"],
    })
    return attrs, df

def _unshuffle_batch(block: bytes, batch_events: int, n_channels: int, trace_samples: int,
                     dtype=np.float16) -> np.ndarray:
    dtype = np.dtype(dtype)
    itemsize = dtype.itemsize
    num_elements = n_channels * trace_samples
    u8 = np.frombuffer(block, dtype=np.uint8)
    expected = batch_events * itemsize * num_elements
    if u8.size != expected:
        raise ValueError(f"Unexpected batch size: got {u8.size} bytes, expected {expected}")
    u8 = (u8.reshape(batch_events, itemsize, num_elements)
             .swapaxes(1, 2)
             .reshape(batch_events, num_elements * itemsize))
    arr = u8.view(dtype)  # (B, N)
    return arr.reshape(batch_events, n_channels, trace_samples)

def iter_traces_zst_batched_merged(traces_path: Path,
                                   n_events: int,
                                   n_channels: int,
                                   trace_samples: int,
                                   batch_size: int = 200,
                                   dtype=np.float16,
                                   max_events: int | None = None):
    dtype = np.dtype(dtype)
    per_event_bytes = int(n_channels * trace_samples * dtype.itemsize)
    to_read = n_events if max_events is None else min(max_events, n_events)

    dctx = zstd.ZstdDecompressor()
    with open(traces_path, "rb") as fin, dctx.stream_reader(fin) as reader:
        buf = io.BufferedReader(reader)
        remaining = to_read
        while remaining > 0:
            bsz = int(min(batch_size, remaining))
            need = per_event_bytes * bsz
            got, chunks = 0, []
            while got < need:
                chunk = buf.read(need - got)
                if not chunk:
                    raise EOFError(f"Unexpected end of stream: need {need} bytes, got {got} bytes")
                chunks.append(chunk)
                got += len(chunk)
            block = b"".join(chunks)
            yield _unshuffle_batch(block, bsz, n_channels, trace_samples, dtype=dtype)
            remaining -= bsz

def open_merged_dataset(base_dir: Path, energy: int):
    base = Path(base_dir)
    meta_path = base / f"meta_energy_{energy}.h5"
    traces_path = base / f"traces_energy_{energy}.zst"
    attrs, meta_df = read_meta_h5(meta_path)
    n_events = len(meta_df)
    n_channels = int(attrs["n_channels"])
    trace_samples = int(attrs["trace_samples"])
    trace_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

    def get_iter(batch_size=200, dtype=trace_dtype, max_events=None):
        return iter_traces_zst_batched_merged(traces_path, n_events, n_channels, trace_samples,
                                              batch_size=batch_size, dtype=dtype, max_events=max_events)
    return attrs, meta_df, get_iter

# -----------------------------
# (2) IterableDataset
# -----------------------------
class MergedTrainingDataset(IterableDataset):
    """
    Streams batches from a single merged .zst file and aligns with metadata rows.
    Yields dicts:
      - "x": (B, C, T) float32
      - "mask": (B, C) float32
      - "pos": (B, 3) float32
      - "energy": (B,) float32
      - "cls": (B,) int64
    """
    def __init__(self, base_dir: Path, energy: int, batch_size: int = 64,
                 out_dtype=np.float32, max_events: int | None = None):
        super().__init__()
        self.base_dir = Path(base_dir)
        self.energy = energy
        self.batch_size = int(batch_size)
        self.out_dtype = np.dtype(out_dtype)
        self.max_events = max_events

        attrs, meta_df, get_iter = open_merged_dataset(self.base_dir, self.energy)
        self.attrs = attrs
        self.meta_df = meta_df.reset_index(drop=True)
        self.get_iter = get_iter

        self.n_events = len(self.meta_df)
        self.n_channels = int(attrs["n_channels"])
        self.trace_samples = int(attrs["trace_samples"])
        self.in_dtype = np.dtype(attrs.get("trace_dtype", np.float16))

        classes = sorted(self.meta_df["type_recoil"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.num_classes = len(classes)

    def __iter__(self):
        wi = get_worker_info()
        if wi is not None and wi.num_workers > 1 and wi.id != 0:
            return iter(())  # only worker 0 reads the single stream

        meta_idx = 0
        for np_batch in self.get_iter(batch_size=self.batch_size, dtype=self.in_dtype, max_events=self.max_events):
            B = np_batch.shape[0]

            if self.out_dtype != np_batch.dtype:
                np_batch = np_batch.astype(self.out_dtype, copy=False)
            x = torch.from_numpy(np_batch)  # (B, C, T)

            mask = torch.ones((B, self.n_channels), dtype=torch.float32)

            meta_slice = self.meta_df.iloc[meta_idx: meta_idx + B]
            meta_idx += B

            pos = torch.tensor(meta_slice[["x", "y", "z"]].to_numpy(), dtype=torch.float32)
            energy = torch.tensor(meta_slice["energy"].to_numpy(), dtype=torch.float32)
            cls = torch.tensor([self.class_to_idx[s] for s in meta_slice["type_recoil"]], dtype=torch.long)

            yield {"x": x, "mask": mask, "pos": pos, "energy": energy, "cls": cls}

# -----------------------------
# (3) Training utils
# -----------------------------
def human_mb(t: torch.Tensor) -> float:
    return (t.element_size() * t.nelement()) / (1024**2)

@torch.no_grad()
def _trim_cache_every(n, step):
    if step % n == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

def assert_model_on(device, model: nn.Module):
    for n, p in model.named_parameters(recurse=True):
        if p is not None and p.data.numel() and p.device != device:
            raise RuntimeError(f"Param {n} on {p.device}, expected {device}")
    for n, b in model.named_buffers(recurse=True):
        if b is not None and b.data.numel() and b.device != device:
            raise RuntimeError(f"Buffer {n} on {b.device}, expected {device}")

def train_one_epoch(model, loader, optimizer, scaler, device,
                    w_pos=1.0, w_energy=1.0, w_cls=1.0,
                    accum_steps=4, cache_trim_every=50):
    """Gradient accumulation to reduce peak VRAM."""
    model.train()
    loss_pos_fn = nn.SmoothL1Loss()
    loss_energy_fn = nn.MSELoss()
    loss_cls_fn = nn.CrossEntropyLoss()

    optimizer.zero_grad(set_to_none=True)
    steps = 0

    for batch in loader:
        # CPU tensors
        x_cpu = batch["x"]; m_cpu = batch["mask"]
        yp_cpu = batch["pos"]; ye_cpu = batch["energy"]; yc_cpu = batch["cls"]
        del batch

        B = x_cpu.shape[0]
        mb = max(1, B // accum_steps)
        if B % mb != 0:
            mb = 1

        for i in range(0, B, mb):
            x = x_cpu[i:i+mb].to(device, non_blocking=True)
            m = m_cpu[i:i+mb].to(device, non_blocking=True)
            yp = yp_cpu[i:i+mb].to(device, non_blocking=True)
            ye = ye_cpu[i:i+mb].to(device, non_blocking=True)
            yc = yc_cpu[i:i+mb].to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
                pos_pred, energy_pred, cls_logit, aux = model(x, channel_mask=m)
                if energy_pred.dim() == 2 and energy_pred.shape[1] == 1:
                    energy_pred = energy_pred.squeeze(1)

                loss_pos = loss_pos_fn(pos_pred, yp)
                loss_energy = loss_energy_fn(energy_pred, ye)
                loss_cls = loss_cls_fn(cls_logit, yc)
                loss = w_pos * loss_pos + w_energy * loss_energy + w_cls * loss_cls
                loss = loss * (mb / B)  # scale for accumulation

            scaler.scale(loss).backward()

            # free microbatch temporaries
            del x, m, yp, ye, yc, pos_pred, energy_pred, cls_logit, aux, loss_pos, loss_energy, loss_cls, loss

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        steps += 1
        _trim_cache_every(cache_trim_every, steps)

        # free CPU batch tensors
        del x_cpu, m_cpu, yp_cpu, ye_cpu, yc_cpu

    return {}

# -----------------------------
# (4) Main
# -----------------------------
if __name__ == "__main__":
    from tcxformer_v2 import ModelConfig, ModalityTCXFormer  # your file

    torch.manual_seed(0)

    # ---- Choose GPU explicitly and make it the default tensor device
    gpu_index = 0
    if torch.cuda.is_available():
        torch.cuda.set_device(gpu_index)                          # set default CUDA device
        device = torch.device(f"cuda:{gpu_index}")
        try:
            # PyTorch >= 2.0: make all new tensors default to this GPU
            torch.set_default_device(device)
        except Exception:
            pass
    else:
        device = torch.device("cpu")
    print("Using device:", device)

    # ---- Model
    cfg = ModelConfig(
        n_photon=19, n_phonon=37,
        stride_photon=128, stride_phonon=256,
        d_model=256, d_ff=1024, n_heads=4,
        n_time_layers=3, n_chan_layers=1,
        patch_embed=64, dropout=0.1,
        rope_base=10000.0, n_branch_to_task=2,
    )
    model = ModalityTCXFormer(cfg).to(device)

    # sanity check: params & buffers are on the right device
    assert_model_on(device, model)

    # ---- Data
    BASE = Path("/ceph/dwong/work/training_samples/ER/small")
    ENERGY_BIN = 1000
    ds = MergedTrainingDataset(BASE, ENERGY_BIN, batch_size=64, out_dtype=np.float32, max_events=None)
    print(f"Classes: {ds.class_to_idx}  (num={ds.num_classes})")

    loader = DataLoader(
        ds,
        batch_size=None,       # dataset yields ready-made batches
        num_workers=0,         # keep 0/1 for single merged .zst
        pin_memory=False,
        persistent_workers=False,
        prefetch_factor=None,
    )

    # ---- Optimizer / AMP
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    try:
        for epoch in range(1, 4):
            stats = train_one_epoch(
                model, loader, optimizer, scaler, device,
                w_pos=1.0, w_energy=1.0, w_cls=1.0,
                accum_steps=16,
                cache_trim_every=50
            )
            print(f"[epoch {epoch}] done")
    finally:
        # Aggressive cleanup (useful in notebooks/REPL)
        del model, optimizer, scaler, loader, ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


Using device: cuda:2
Classes: {'ER': 0}  (num=1)


/tmp/ipykernel_3467185/1048218514.py:289: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_3467185/1048218514.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
/tmp/ipykernel_3467185/1048218514.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
/tmp/ipykernel_3467185/1048218514.py:208: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
/tmp/ipykernel_3467185/1048218514.py:208: FutureWarning: `torch.cuda.amp.auto

In [6]:
a = 1063252850

In [7]:
a=a/56

In [8]:
a

18986658.035714287

In [5]:
a

126.57772023809525